# P3 — Paths are messages

The bridge between P1 (over-squashing) and P2 (counting paths). After `g` layers, a message from `i` reaches
`j` **once per length-`g` path**. So the path count `n_g(i,j)` is exactly *how many copies of `i`'s
information land at `j`* — all crammed into `j`'s one vector.

**5 figures:** the message stack vs capacity · explosion · raw-vs-deduplicated bars · the two operators ·
redundant paths merging.

In [ ]:
import torch, numpy as np
from oversquash import viz
from oversquash.data_bottleneck import make_bottleneck_graph
from oversquash.ideal_bridge import walk_operators
K, M = 5, 4
def msgs_at(depth, op='raw'):
    dd = make_bottleneck_graph(K, M, depth, torch.Generator().manual_seed(0))
    raw, eff = walk_operators(dd.edge_index.numpy(), dd.num_nodes, max_length=depth+1)
    t = int(dd.root_mask.nonzero()[0])
    return (raw if op=='raw' else eff)[depth][:, t].sum()

## Figure 1 — messages stacking past the vector's capacity

Each bar is the number of messages arriving at the target. The dashed line is the vector's capacity. Past it,
the bar turns red — those messages can't fit. *That* is the squash.

In [ ]:
depths = [1,2,3,4]
raw_msgs = [int(msgs_at(d, 'raw')) for d in depths]
viz.explain_message_stack(raw_msgs, [f'depth {d}' for d in depths], capacity=10,
                          title='messages at the target vs. vector capacity', xlabel='')

## Figure 2 — the explosion, on a log scale

Same counts, log axis: `K·M^depth` copies arrive. The vector size never grows to match.

In [ ]:
viz.plot_message_explosion(depths, raw_msgs, 'messages arriving at the target', 'depth', 'messages (log)')

## Figure 3 — but many messages are redundant

In our bottleneck, the middle nodes are interchangeable, so many paths carry the **same** content. The quotient
`kQ/I` merges equivalent paths: the count collapses from `K·M^depth` to ~`K`.

In [ ]:
eff_msgs = [int(msgs_at(d, 'eff')) for d in depths]
viz.plot_raw_vs_eff(depths, raw_msgs, eff_msgs, 'raw vs de-duplicated messages', 'depth', 'messages (log)')

## Figure 4 — the two redundant paths, side by side

Two interchangeable routes carrying the same content. `kQ/I` recognizes they are equivalent and counts them
once. (Drawn on a small diamond for clarity.)

In [ ]:
dia = [(0,1),(0,2),(1,3),(2,3)]; pos = {0:(0,0),1:(1,1),2:(1,-1),3:(2,0)}
viz.explain_paths_highlighted(dia, pos, [[(0,1),(1,3)],[(0,2),(2,3)]], 4, 0, 3,
                              title='two redundant routes -> kQ/I keeps one', path_fmt='route {i}')

## Figure 5 — the operators: raw vs de-duplicated

`A^g` (raw walk counts) vs `M_g` (kQ/I de-duplicated). Same support, far fewer copies on the right.

**The plot twist (proved in P4–P5):** merging is the *wrong* fix — it throws away signal. The right fix is to
keep all paths but **learn how much to weight each** (attention).

In [ ]:
dd = make_bottleneck_graph(K, M, 3, torch.Generator().manual_seed(0))
raw, eff = walk_operators(dd.edge_index.numpy(), dd.num_nodes, max_length=4)
viz.plot_two_heatmaps(raw[3], eff[3], titles=('A^4 (raw)', 'M_4 (kQ/I, de-duplicated)'))